# 原始SEC申报EPS文本核查
两个季度原始申报附件与公司报告交叉；保留实际读取文本，不把渲染文本称为原始HTML。

In [1]:
from pathlib import Path
import os,json,hashlib
import qlib,pandas as pd
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/nvda_sec_eps_pilot_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/nvda_eps_corpus_20261015_v1';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
corpus,audit=_source(root,p['corpus_source'],['analysis/events']);events={e['fiscal_quarter']:e for e in corpus['analysis/events']};raw={}
if all((root/p['raw_dir']/e['file']).exists() for e in p['payloads']):raw={e['file']:(root/p['raw_dir']/e['file']).read_bytes() for e in p['payloads']}
else:
 old=json.loads((card/'summary.json').read_text())
 if not Path(old['recorder_local_dir']).exists():materialize_nas_evidence(card/'nas_binding.json',card=card)
 with R.uri_context(str(card/'qlib_recorder')):raw=R.get_recorder(recorder_id=old['recorder_id'],experiment_id=old['experiment_id']).load_object('inputs/web_text_bytes')
for e in p['payloads']:assert hashlib.sha256(raw[e['file']]).hexdigest()==e['sha256']
rows=[]
for case in p['cases']:
 text=raw[case['file']].decode();assert case['url'] in text and case['required_text'] in text;record=events[case['quarter']];assert record['current_eps']==case['eps'] and record['prior_year_comparative_eps']==case['prior_year_eps'];rows.append({'quarter':case['quarter'],'accession':case['accession'],'eps':case['eps'],'prior_year_comparative_eps':case['prior_year_eps'],'source_url':case['url'],'current_original_accession_text_crosschecked':True,'exact_web_text_retained':True,'original_html_bytes_retained':False,'first_publication_clock_verified':False,'training_input_qualified':False})
table=pd.DataFrame(rows);policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/verified_facts']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,web_text_bytes=raw,corpus_audit=audit);rec.save_objects(artifact_path='analysis',verified_facts=table);rec.log_metrics(verified_quarters=2,new_fits=0,new_backtests=0)
summary={'experiment':card.name,'decision':'needs_continuous_original_filing_series_and_clock','new_fits':0,'new_backtests':0,'verified_quarters':len(table),'training_input_qualified':False,'provider_uri':str(provider),'region':'us','as_of':'2026-10-14','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':p['scope']};rec.save_objects(artifact_path='analysis',summary=summary);a=Path(summary['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in a.rglob('*')));table.to_csv(card/'verified_quarters.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table.to_string(index=False))


# SEC原始申报EPS核查原型：两季度文本已留存，连续训练序列仍缺

本轮未训练、未回测。公司事实API、submissions刷新及2016Q2财务报表ZIP直接下载均返回403；不能把本轮请求失败说成来源不存在。官方说明companyfacts按申报聚合，frames偏向最后申报匹配期间，后者不能直接作为当时版本。财务报表批次在2024年重新处理过，当前下载批次也不能自动当成2016年的归档字节。

改为使用已有申报编号，读取两个SEC附件：FY2017Q1当前GAAP diluted EPS0.33、上年比较0.24；FY2018Q2当前0.92、上年比较0.41。标题、GAAP与Non-GAAP区分、正文和表格均经检查，与已有公司报告语料数值一致。0.41是后来报告中的上年比较值，不替换FY2017Q2最初的0.40。FY2017Q2、FY2018Q1两个索引此次读取报内部错误，未据命名猜附件内容。

原始来源链接：
- https://www.sec.gov/Archives/edgar/data/1045810/000104581016000243/q117pr.htm
- https://www.sec.gov/Archives/edgar/data/1045810/000104581017000117/q2fy18pr.htm

已保存web工具实际返回的完整文本响应及SHA256，含来源URL与行定位，原始内容存Qlib Recorder/NAS；不是只记忆一个链接或手写数值。表示层明确为渲染文本，不是原始HTML，不宣称首次披露字节、精确首次发布时间或完整申报正文均已认证。工具读取成功也不等于本地HTTP下载成功。已有submissions元数据仍保持此前版本身份，未将失败刷新替换原数据。

当前只有2个不连续季度的原始申报文本核对，不能计算连续历史要求的SUE，更不能把单一发行人代理叫全SOXX盈利预期。源报表、后来可比值、文件申报时间和首发公告时点继续分开；不猜UTC/美东转换，不用延迟假设代替版本核对。

结论needs_continuous_original_filing_series_and_clock，training_input_qualified=false。缺连续原始申报数值和可用时间证据；不是已完成特征训练，也不是超额改进。下一步为补齐剩余季度的原始附件和时间绑定，不重跑已关闭的一致预期或指引路线。生产模型及原账户收益不变，稳定超额目标未完成。

notebook执行核对来源哈希及与父Recorder数值一致，NAS冷恢复逐对象哈希通过。本轮无新增业务逻辑，不增加测试套件。官方接口语义：https://www.sec.gov/search-filings/edgar-application-programming-interfaces；季度批次说明：https://www.sec.gov/data-research/sec-markets-data/financial-statement-data-sets。
